# Evaluating QueryBot with pydantic_evals

Demonstrates writing a `pydantic_evals` `Dataset` against a live QueryBot: a task
function that runs one question through a fresh bot, `Evaluator`s that inspect the
response status and trace, and a `pass_rate()` helper that repeats the dataset several
times for confidence, since a live model is non-deterministic.

This is the live-model companion to `tests/evals/` (this repo's CI-safe reference
harness, driven by scripted `FunctionModel`s with no live API calls). Point this
notebook's pattern at your own spec catalog and questions to measure whether your agent
actually behaves the way you expect — `tests/evals/` only proves the trace/result-store/
evaluator wiring is consumable, not that any given model selects the right tool or
refuses appropriately.

Prerequisites
-------------
1. Create the DuckDB database (one-time setup): `python examples/data/setup_db.py`
2. Set your Anthropic API key: `export ANTHROPIC_API_KEY=sk-ant-...`
3. Install the agent-evals extra: `pip install aitaem[agent-evals]`

**Cost/runtime note:** `pass_rate(n=5)` below runs the 2-case dataset 5 times against a
live model — 10 bot invocations total. Each `bot.ask()` call is a multi-step tool loop
(`record_intent` → `resolve_intent` → `compute_metrics`, each a separate model turn), so
the actual number of model calls is meaningfully higher than 10. Budget accordingly
before running that cell.

In [1]:
from __future__ import annotations

import os
from collections import Counter
from dataclasses import dataclass

from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Evaluator, EvaluatorContext

from aitaem.connectors import ConnectionManager
from aitaem.specs import SpecCache
from aitaem.agent import QueryBot, QueryResponse, Status

MODEL = "anthropic:claude-haiku-4-5-20251001"


def _check_api_key() -> str:
    key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not key:
        raise RuntimeError(
            "ANTHROPIC_API_KEY is not set. "
            "Export it before running: export ANTHROPIC_API_KEY=sk-ant-..."
        )
    return key

In [2]:
# ---------------------------------------------------------------------------
# Task + I/O types
# ---------------------------------------------------------------------------

@dataclass
class QIn:
    question: str


@dataclass
class QOut:
    response: QueryResponse   # aitaem.agent.QueryResponse
    bot: QueryBot             # kept so evaluators can call get_result()


async def query_task(inputs: QIn) -> QOut:
    # Fresh bot per case — per-run state (intents, spec registry) must not leak
    # across cases. SPEC_CACHE/CONN_MGR are safe to share: read-only catalog
    # and connection, set up below before the dataset runs.
    bot = QueryBot(
        model=MODEL,
        spec_cache=SPEC_CACHE,
        connection_manager=CONN_MGR,
    )
    response = await bot.ask(inputs.question)  # single-turn, no history
    return QOut(response=response, bot=bot)

In [ ]:
# ---------------------------------------------------------------------------
# Evaluators
# ---------------------------------------------------------------------------

@dataclass
class StatusIs(Evaluator[QIn, QOut, None]):
    expected: Status

    def evaluate(self, ctx: EvaluatorContext[QIn, QOut, None]) -> bool:
        return ctx.output.response.status is self.expected


@dataclass
class CalledTool(Evaluator[QIn, QOut, None]):
    tool_name: str

    def evaluate(self, ctx: EvaluatorContext[QIn, QOut, None]) -> bool:
        return any(
            tc.name == self.tool_name for tc in ctx.output.response.trace.tool_calls
        )


@dataclass
class ToolSequenceIs(Evaluator[QIn, QOut, None]):
    """Exact-sequence check — the one that actually tests the Metric Precision
    Rule: did the bot gate through record_intent -> resolve_intent before
    calling compute_metrics, in exactly that order?

    Deliberately strict, not a subsequence check. Against a live model, a run
    that self-corrects (e.g. calls resolve_intent twice after an initial
    near-miss) behaves correctly — it still gated before computing — but
    fails this exact-sequence assertion. A subsequence check would fix that,
    but would also let a model that calls compute_metrics before
    resolve_intent pass, as long as it calls resolve_intent *later* in the
    same run — defeating the one case that exercises the gate order at all.
    A single failed run on this assertion isn't necessarily a bug; read it
    through pass_rate() below, not a single run's pass/fail.

    Note the trailing "final_result" in the expected list below: pydantic-ai's
    default structured-output mechanism for a plain `output_type=` (as
    QueryBot uses) is a synthetic tool call named "final_result"
    (pydantic_ai._output.DEFAULT_OUTPUT_TOOL_NAME) — every successful run ends
    with one. Omitting it here would make this assertion fail on every real
    run, not just occasionally.
    """

    expected: list[str]

    def evaluate(self, ctx: EvaluatorContext[QIn, QOut, None]) -> bool:
        return [tc.name for tc in ctx.output.response.trace.tool_calls] == self.expected

In [ ]:
# ---------------------------------------------------------------------------
# Dataset — two cases against the real examples/metrics/ catalog. Neither
# question names a metric in that catalog, so the refusal case is genuine,
# not contrived.
# ---------------------------------------------------------------------------

dataset: Dataset[QIn, QOut, None] = Dataset(
    name="query_bot_behavioral_eval",
    cases=[
        Case(
            name="in_catalog_metric",
            inputs=QIn("What was total revenue in Q1 2024?"),
            evaluators=(
                StatusIs(Status.ok),
                ToolSequenceIs(
                    ["record_intent", "resolve_intent", "compute_metrics", "final_result"]
                ),
            ),
        ),
        Case(
            name="out_of_catalog_metric",
            inputs=QIn("What was sales velocity last month?"),
            evaluators=(
                StatusIs(Status.refused),
                CalledTool("resolve_intent"),  # it must try, then refuse
            ),
        ),
    ]
)

In [5]:
# ---------------------------------------------------------------------------
# Repeated-run confidence — a live model is non-deterministic, so a single
# dataset.evaluate() run tells you much less than a distribution does.
# ---------------------------------------------------------------------------

async def pass_rate(dataset: Dataset[QIn, QOut, None], task, n: int = 5) -> dict[str, float]:
    tally: Counter[str] = Counter()
    for _ in range(n):
        report = await dataset.evaluate(task)
        for case in report.cases:
            ok = all(assertion.value is True for assertion in case.assertions.values())
            tally[case.name] += ok
    return {name: count / n for name, count in tally.items()}

Set the path to the local folder which contains your AITAEM examples. The following
sub-folders are needed inside `aitaem_folder_path`:
- `examples/data`
- `examples/metrics`
- `examples/slices`

In [ ]:
aitaem_folder_path = "/path/to/aitaem"  # e.g. "/Users/you/Workspace/aitaem"

In [ ]:
# ---------------------------------------------------------------------------
# Setup — spec catalog + DuckDB connection, shared (read-only) across cases.
# ---------------------------------------------------------------------------

_check_api_key()

SPEC_CACHE = SpecCache.from_yaml(
    metric_paths=aitaem_folder_path + "/examples/metrics/",
    slice_paths=aitaem_folder_path + "/examples/slices/",
)
print(f"{len(SPEC_CACHE.metrics)} metrics: {', '.join(SPEC_CACHE.metrics)}")

db_path = aitaem_folder_path + "/examples/data/ad_campaigns.duckdb"
if not os.path.exists(db_path):
    print("DuckDB file not found — creating from CSV …")
    from aitaem.helpers import load_csvs_to_duckdb
    csv_path = aitaem_folder_path + "/examples/data/ad_campaigns.csv"
    load_csvs_to_duckdb(csv_path, db_path)
    print(f"Created {db_path}")

CONN_MGR = ConnectionManager()
CONN_MGR.add_connection("duckdb", path=db_path)

## Run the dataset once

A single run against a live model — useful for a quick look, but see the note on
`ToolSequenceIs` above before reading a single failed assertion as a bug.

In [ ]:
report = await dataset.evaluate(query_task)
report.print(include_input=True, include_output=False)

## Repeated-run confidence

10 bot invocations (5 reps × 2 cases), each a multi-turn tool loop — meaningfully more
than 10 model calls. Budget accordingly before running this cell.

In [ ]:
rates = await pass_rate(dataset, query_task, n=5)
for name, rate in rates.items():
    print(f"{name}: {rate:.0%}")